# Prompt engineering

Improve a prompt in a loop: **set a goal → write a first prompt → evaluate →
apply one technique → re-evaluate**, and repeat the last two steps.
Change one thing at a time so you know what helped.

Example task: a one-day meal plan for an athlete from height, weight, goal and dietary restrictions.

`PromptEvaluator` (in `prompt_evaluator.py`, taken from the course notebook) does the work:

- `generate_dataset(task_description, prompt_inputs_spec, output_file, num_cases)`:
  Claude first invents distinct test ideas, then turns each into a test case with
  `prompt_inputs` and `solution_criteria`
- `run_evaluation(run_prompt_function, dataset_file, extra_criteria)`: runs the prompt on every case
  in parallel, grades each output with a model, prints the average and writes `output.json` +
  `output.html`
- `extra_criteria` are mandatory: missing one caps the score at 3
- `max_concurrent_tasks`: start low (3) to avoid rate limits

A low first score (the course mentions 2.3) is normal: it is the baseline to beat.

## Setup

In [1]:
import os

from prompt_evaluator import PromptEvaluator, add_messages, chat

evaluator = PromptEvaluator(max_concurrent_tasks=3)

## Dataset

Generated once and kept in `dataset.json`, so every prompt version is graded on the same cases.
Delete the file to generate a new one. Keep the number of cases low (2-3) while iterating.

In [2]:
if not os.path.exists("dataset.json"):
    evaluator.generate_dataset(
        task_description="Write a compact, concise 1 day meal plan for a single athlete",
        prompt_inputs_spec={
            "height": "Athlete's height in cm",
            "weight": "Athlete's weight in kg",
            "goal": "Goal of the athlete",
            "restrictions": "Dietary restrictions of the athlete",
        },
        output_file="dataset.json",
        num_cases=3,
    )

## Baseline prompt

Deliberately naive: it does not say what a good answer looks like.

In [3]:
def run_prompt(prompt_inputs):
    prompt = f"""
What should this person eat?
- Height: {prompt_inputs["height"]}
- Weight: {prompt_inputs["weight"]}
- Goal: {prompt_inputs["goal"]}
- Dietary restrictions: {prompt_inputs["restrictions"]}
"""
    messages = []
    add_messages(messages, prompt)
    return chat(messages)

## Evaluate

In [4]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset.json",
    extra_criteria="""
The output should include:
- Daily caloric total
- Macronutrient breakdown
- Meals with exact foods, portions, and timing
""",
)

Graded 1/3 test cases
Graded 2/3 test cases


Graded 3/3 test cases
Average score: 2.3333333333333335


In [5]:
for result in results:
    print(f"\n[{result['score']}] {result['test_case']['scenario']}")
    print(f"    {result['reasoning']}")


[3] Testing with a high-endurance athlete (marathon runner) requiring calorie-dense meals and carbohydrate loading strategy
    The solution fails to meet mandatory requirements. While it provides general targets and a meal framework, it does not deliver a specific 1-day meal plan with exact foods, portions, and timing as required. The solution reads as a template/guide rather than an actionable daily plan. Additionally, the caloric and carbohydrate targets are insufficient for the stated criteria (should be 3,500-4,000 calories and 490-700g carbs for a 70kg marathon runner). The solution ends with 'Would you like specific meal examples?' which confirms it did not provide the required specificity. A proper response would have included something like: 'Breakfast: 2 cups oatmeal (100g carbs), 1 banana (27g carbs), 2 tbsp honey (34g carbs)...' with a complete daily total.

[2] Testing with a strength-focused athlete (weightlifter) emphasizing protein intake and meal timing around workout

Open `output.html` for the full report: inputs, criteria, output and reasoning per case.